### Imports

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
from torchvision import transforms
import os


### Configurations

In [11]:
# Configuration
# Configuration - CORRECTED PATHS
CHEXPERT_CSV_PATH = '/home/cadenug/ML Final Project/datasets/ashery/chexpert/versions/1/'
CHEXPERT_IMG_PATH = '/home/cadenug/ML Final Project/datasets/ashery/chexpert/versions/1/'


SELECTED_DISEASES = [
    'Lung Opacity',
    'Pleural Effusion', 
    'Atelectasis',
    'Pneumothorax',
    'Pneumonia'
]


### CheXpert Datastes Class

In [12]:

class CheXpertDataset(Dataset):
    """CheXpert Dataset with U-Zeros strategy"""
    
    def __init__(self, dataframe, root_dir, transform=None):
        """
        Args:
            dataframe: Pandas DataFrame with image paths and labels
            root_dir: Root directory for images
            transform: Torchvision transforms
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.diseases = SELECTED_DISEASES
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # Get image path
        img_path = self.dataframe.iloc[idx]['Path']
        
        # Strip 'CheXpert-v1.0-small/' prefix and construct full path
        img_path = img_path.replace('CheXpert-v1.0-small/', '')
        full_path = os.path.join(self.root_dir, img_path)
        
        # Load image
        try:
            image = Image.open(full_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {full_path}: {e}")
            # Return a blank image if load fails
            image = Image.new('RGB', (224, 224))
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get labels (multi-hot encoded)
        labels = []
        for disease in self.diseases:
            label_value = self.dataframe.iloc[idx][disease]
            labels.append(label_value)
        
        labels = torch.FloatTensor(labels)
        
        return image, labels



### Load Dataloaders

In [13]:
def load_chexpert_data(batch_size=32, num_workers=4):
    """
    Load and prepare CheXpert dataloaders
    
    Args:
        batch_size: Batch size for training
        num_workers: Number of workers for data loading
        
    Returns:
        train_loader, val_loader, train_dataset, val_dataset
    """
    
    print("="*80)
    print("LOADING CHEXPERT DATA")
    print("="*80)
    
    # Load CSVs
    train_df = pd.read_csv(os.path.join(CHEXPERT_CSV_PATH, 'train.csv'))
    val_df = pd.read_csv(os.path.join(CHEXPERT_CSV_PATH, 'valid.csv'))
    
    print(f"\n📊 Original train samples: {len(train_df):,}")
    print(f"📊 Original val samples: {len(val_df):,}")
    
    # Filter to Frontal views only
    train_df = train_df[train_df['Frontal/Lateral'] == 'Frontal'].copy()
    val_df = val_df[val_df['Frontal/Lateral'] == 'Frontal'].copy()
    
    print(f"\n📊 After filtering to Frontal only:")
    print(f"   Train: {len(train_df):,}")
    print(f"   Val: {len(val_df):,}")
    
    # Apply U-Zeros strategy: -1.0 → 0.0, NaN → 0.0
    print(f"\n⚙️  Applying U-Zeros strategy...")
    for disease in SELECTED_DISEASES:
        # Fill NaN with 0.0
        train_df[disease] = train_df[disease].fillna(0.0)
        val_df[disease] = val_df[disease].fillna(0.0)
        
        # Replace -1.0 with 0.0
        train_df[disease] = train_df[disease].replace(-1.0, 0.0)
        val_df[disease] = val_df[disease].replace(-1.0, 0.0)
    
    print(f"✅ U-Zeros applied to all {len(SELECTED_DISEASES)} diseases")
    
    # Check class distribution
    print(f"\n📊 Class Distribution (Train Set):")
    print(f"{'Disease':<20} {'Positive':>10} {'Negative':>10} {'% Positive':>12}")
    print("-" * 55)
    for disease in SELECTED_DISEASES:
        pos = (train_df[disease] == 1.0).sum()
        neg = (train_df[disease] == 0.0).sum()
        pct = (pos / len(train_df)) * 100
        print(f"{disease:<20} {pos:>10,} {neg:>10,} {pct:>11.2f}%")
    
    # Define transforms
    # ImageNet normalization (standard for pretrained models)
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),  # Data augmentation
        transforms.RandomRotation(10),            # Slight rotation
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = CheXpertDataset(
        dataframe=train_df,
        root_dir=CHEXPERT_IMG_PATH,
        transform=train_transform
    )
    
    val_dataset = CheXpertDataset(
        dataframe=val_df,
        root_dir=CHEXPERT_IMG_PATH,
        transform=val_transform
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True  # Faster GPU transfer
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    print(f"\n✅ DataLoaders created:")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    print(f"   Batch size: {batch_size}")
    
    return train_loader, val_loader, train_dataset, val_dataset


# Calculate class weights for Weighted BCE
def calculate_class_weights(train_dataset):
    """Calculate class weights for weighted BCE loss"""
    
    print("\n" + "="*80)
    print("CALCULATING CLASS WEIGHTS FOR WEIGHTED BCE")
    print("="*80)
    
    # Load the dataframe from dataset
    df = train_dataset.dataframe
    total_samples = len(df)
    
    weights = []
    print(f"\n{'Disease':<20} {'Positive':>10} {'Weight':>10}")
    print("-" * 42)
    
    for disease in SELECTED_DISEASES:
        pos_count = (df[disease] == 1.0).sum()
        # Weight = total / (2 * positive_count)
        weight = total_samples / (2 * pos_count) if pos_count > 0 else 1.0
        weights.append(weight)
        print(f"{disease:<20} {pos_count:>10,} {weight:>10.3f}")
    
    weights = torch.FloatTensor(weights)
    print(f"\n✅ Class weights calculated: {weights}")
    
    return weights

### Test the data loader

In [16]:
# Test the data loader
print("Testing data loader...")
train_loader, val_loader, train_dataset, val_dataset = load_chexpert_data(
    batch_size=32,
    num_workers=4
)

# Calculate class weights
class_weights = calculate_class_weights(train_dataset)

print("\n" + "="*80)
print("DATA LOADER TEST")
print("="*80)

# Test loading one batch
images, labels = next(iter(train_loader))
print(f"\n✅ Successfully loaded one batch!")
print(f"   Images shape: {images.shape}")  # Should be [32, 3, 224, 224]
print(f"   Labels shape: {labels.shape}")  # Should be [32, 5]
print(f"   Image range: [{images.min():.2f}, {images.max():.2f}]")
print(f"\n   First sample labels: {labels[0]}")
print(f"   (Order: {SELECTED_DISEASES})")

Testing data loader...
LOADING CHEXPERT DATA

📊 Original train samples: 223,414
📊 Original val samples: 234

📊 After filtering to Frontal only:
   Train: 191,027
   Val: 202

⚙️  Applying U-Zeros strategy...
✅ U-Zeros applied to all 5 diseases

📊 Class Distribution (Train Set):
Disease                Positive   Negative   % Positive
-------------------------------------------------------
Lung Opacity             94,211     96,816       49.32%
Pleural Effusion         76,899    114,128       40.26%
Atelectasis              29,720    161,307       15.56%
Pneumothorax             17,693    173,334        9.26%
Pneumonia                 4,675    186,352        2.45%

✅ DataLoaders created:
   Train batches: 5970
   Val batches: 7
   Batch size: 32

CALCULATING CLASS WEIGHTS FOR WEIGHTED BCE

Disease                Positive     Weight
------------------------------------------
Lung Opacity             94,211      1.014
Pleural Effusion         76,899      1.242
Atelectasis              29,7